# Phase 13 — Evaluation Suite (Gemini-powered RAGAS)

> **Gemini-only project.** Every cell here uses Google Gemini (chat + embeddings) via `langchain-google-genai`. The only key the core needs is `GOOGLE_API_KEY`. This notebook is a **scaffold** — run it top-to-bottom *after* Claude Code finishes this phase. If a cell references a module that doesn't exist yet, that phase hasn't been built.

**Goal:** run a slice of each dataset and confirm RAGAS uses **Gemini**, not OpenAI.

**👤 Optional:** `LANGSMITH_API_KEY` for tracing. No OpenAI key needed anywhere.


In [1]:
# --- Setup: load .env and make the project importable ---
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root
from dotenv import load_dotenv
load_dotenv('../.env', override=True)  # override=True: beat VS Code's stale cached env vars

# This project is Gemini-only. Confirm the one required key is present.
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env (see 00_prerequisites_and_setup.md)'
print('GOOGLE_API_KEY loaded:', bool(os.getenv('GOOGLE_API_KEY')))
print('SEC_USER_AGENT   :', os.getenv('SEC_USER_AGENT', '(set before Phase 5)'))

GOOGLE_API_KEY loaded: True
SEC_USER_AGENT   : iampkumar iampkumar03@gmail.com


## 1. Confirm RAGAS is wired to Gemini (the critical gotcha)

In [2]:
import evals._ragas_compat  # MUST import before any `ragas` import (stubs a module ragas 0.4.3 eagerly imports but langchain-community no longer ships)
from app.llm.factory import get_llm, get_embeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
ragas_llm = LangchainLLMWrapper(get_llm(reasoning=True))   # gemini-3.5-flash
ragas_emb = LangchainEmbeddingsWrapper(get_embeddings())   # gemini-embedding-001
print('RAGAS LLM  ->', ragas_llm.langchain_llm.model)
print('RAGAS EMB  ->', ragas_emb.embeddings.model)
assert 'gemini' in ragas_llm.langchain_llm.model, 'RAGAS must use Gemini'

/home/algo-voyager/Documents/Workspace/Projects/Investment Advisory Agent/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RAGAS LLM  -> gemini-3.5-flash
RAGAS EMB  -> gemini-embedding-001


/tmp/ipykernel_732491/2412238792.py:5: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(get_llm(reasoning=True))   # gemini-3.5-flash
/tmp/ipykernel_732491/2412238792.py:6: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_emb = LangchainEmbeddingsWrapper(get_embeddings())   # gemini-embedding-001


## 2. Run a small eval slice

In [3]:
from evals.harness import run_dataset
results = run_dataset('simple_queries', limit=5)
from evals.metrics.answer_accuracy import score as acc_score
print('accuracy (5 items):', acc_score(results))

{"guard": "pii", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "client_id": "CLT-002", "agent": "input_guard", "session_id": "eval", "level": "info", "timestamp": "2026-07-13T13:55:30.660682Z"}
{"guard": "prompt_injection", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "client_id": "CLT-002", "agent": "input_guard", "session_id": "eval", "level": "info", "timestamp": "2026-07-13T13:55:30.661154Z"}
{"guard": "scope", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "client_id": "CLT-002", "agent": "input_guard", "session_id": "eval", "level": "info", "timestamp": "2026-07-13T13:55:30.661405Z"}
{"query": "What stocks do I own?", "event": "planner_simple", "client_id": "CLT-002", "agent": "planner", "session_id": "eval", "level": "info", "timestamp": "2026-07-13T13:55:30.663137Z"}


AFC is enabled with max remote calls: 10.
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"strategy": "llm", "decision": "portfolio", "event": "route_decision", "client_id": "CLT-002", "agent": "supervisor", "session_id": "eval", "level": "info", "timestamp": "2026-07-13T13:55:38.916067Z"}
{"next_agent": "portfolio", "hop": 1, "event": "supervisor_dispatch", "client_id": "CLT-002", "agent": "supervisor", "session_id": "eval", "level": "info", "timestamp": "2026-07-13T13:55:38.917051Z"}
{"query": "What stocks do I own?", "event": "agent_start", "client_id": "CLT-002", "agent": "portfolio", "session_id": "eval", "level": "info", "timestamp": "2026-07-13T13:55:38.921771Z"}


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"rows": 84, "clients": 10, "event": "portfolios_loaded", "client_id": "CLT-002", "agent": "portfolio", "session_id": "eval", "level": "info", "timestamp": "2026-07-13T13:55:46.583442Z"}


...

AFC is enabled with max remote calls: 10.
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"strategy": "llm", "decision": "END", "event": "route_decision", "client_id": "CLT-005", "agent": "supervisor", "session_id": "eval", "level": "info", "timestamp": "2026-07-13T14:05:15.790954Z"}
{"hops": 1, "event": "supervisor_end", "client_id": "CLT-005", "agent": "supervisor", "session_id": "eval", "level": "info", "timestamp": "2026-07-13T14:05:15.792025Z"}


AFC is enabled with max remote calls: 10.
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 429 Too Many Requests"
Retrying google.genai._api_client.BaseApiClient._request_once in 1.57 seconds as it raised ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 44.029451499s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.google

{"tool": "synthesizer", "error_type": "ChatGoogleGenerativeAIError", "client_id": "CLT-005", "session_id": "eval", "error": "Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. Fo", "event": "tool_error", "agent": "synthesizer", "level": "error", "timestamp": "2026-07-13T14:05:50.863311Z"}
{"event": "safe_exit", "level": "info", "timestamp": "2026-07-13T14:05:50.867581Z"}
{"client_id": "CLT-005", "session_id": "eval", "event": "decision_saved", "level": "info", "timestamp": "2026-07-13T14:05:50.882003Z"}
{"item_id": "simple-05", "latency_s": 58.59, "agents": ["portfolio"], "blocked": true, "error": null, "event": "eval_item_done", "level": "info", "timestamp": "2026-07-13T14:05:50.886064Z"}


ImportError: cannot import name 'score' from 'evals.metrics.answer_accuracy' (/home/algo-voyager/Documents/Workspace/Projects/Investment Advisory Agent/evals/metrics/answer_accuracy.py)

## 3. Routing + guardrail effectiveness

In [ ]:
from evals.metrics.routing_accuracy import score as route_score
from evals.metrics.guardrail_effectiveness import score as guard_score
print('routing :', route_score(results))
adv = run_dataset('adversarial_queries', limit=5)
print('guardrail block rate:', guard_score(adv))

## ✅ Acceptance check

In [ ]:
assert os.getenv('OPENAI_API_KEY') is None, 'No OpenAI key should be needed'
print('Phase 13 acceptance: PASS (RAGAS on Gemini; run `make eval` for the full report)')